

# End-to-End Architectural Decomposition of Qwen3-Omni-30B-A3B

---

## 1. System-Level Overview

Qwen3-Omni is a **natively multimodal Thinker-Talker Mixture-of-Experts (MoE)** architecture engineered to ingest, reason over, and generate across four modalities — **text, image, video, and audio** — within a single, cohesive model. The fundamental design principle is the **strict decoupling** of high-level cognitive reasoning (Thinker) from streaming acoustic waveform synthesis (Talker), while both modules share access to the full conversational history and operate over a unified representational space.

### 1.1 Key Architectural Departures from Qwen2.5-Omni

| Design Axis | Qwen2.5-Omni | Qwen3-Omni |
|---|---|---|
| Thinker / Talker backbone | Dense Transformer | **MoE Transformer** (both modules) |
| Talker conditioning source | Thinker's high-level text representations + multimodal features | **Only audio and visual multimodal features** (text representations decoupled) |
| System prompt control | Shared | **Independent** system prompts for Thinker (response style) and Talker (audio style) |
| Codec generation | Block-context DiT-based | **Left-context-only multi-codebook autoregressive** (MTP) |
| Final waveform decoder | DiT-based vocoder | **Lightweight causal ConvNet** (Code2Wav) |
| First-packet synthesis trigger | Must wait for sufficient block-context from Talker | **Immediate** waveform output after each Talker token |

### 1.2 Motivation for Talker Decoupling from Thinker Text Representations

Two principled justifications underpin this design:

1. **Information equivalence**: For textual content, discrete tokens and their corresponding embeddings are effectively information-equivalent; passing high-level text representations from Thinker to Talker is redundant.
2. **Multimodal conditioning necessity**: Audio-video-coordinated speech generation (e.g., preserving prosody and timbre during speech translation) requires direct conditioning on raw multimodal features, not abstracted text representations.

**Practical consequence**: External modules — RAG pipelines, function-calling engines, safety filters — can intervene on the Thinker's textual output independently. If desired, modified text can be supplied to the Talker via controlled preprocessing for streaming synthesis, without disrupting the acoustic generation pathway.

### 1.3 Module Parameter Summary

| Module | Architecture | Parameters | Streaming Support |
|---|---|---|---|
| Audio Encoder (AuT) | Attention Encoder-Decoder | $\approx 650\text{M}$ | $\checkmark$ |
| Vision Encoder | SigLIP2-So400M | $\approx 540\text{M}$ | $-$ |
| Thinker | MoE Transformer | $30\text{B total}$, $3\text{B active}$ | $\checkmark$ |
| Talker | MoE Transformer | $3\text{B total}$, $0.3\text{B active}$ | $\checkmark$ |
| MTP Module | Dense Transformer | $\approx 80\text{M}$ | $\checkmark$ |
| Code2Wav | Causal ConvNet | $\approx 200\text{M}$ | $\checkmark$ |

$$
\text{Total Parameters} \approx 650\text{M} + 540\text{M} + 30\text{B} + 3\text{B} + 80\text{M} + 200\text{M}
$$

---

## 2. Multimodal Input Processing and Perceivation (Raw Inputs)

The ingestion pipeline **strictly harmonizes** heterogeneous raw data streams into an aligned, high-dimensional representational space before routing to the MoE layers. Every modality undergoes a dedicated encoding pathway, and all resulting representations are temporally aligned before fusion.

---

### 2.1 Text Modality

**Raw Input**: Arbitrary Unicode text string $X_{\text{text}}$.

**Step 1 — Tokenization**: A **byte-level byte-pair encoding (BPE)** algorithm segments the input string into a sequence of sub-word tokens. The vocabulary comprises:

$$
|V| = 151{,}643 \quad \text{regular tokens}
$$

Byte-level BPE guarantees **zero out-of-vocabulary failures**: any byte sequence is representable, providing robustness to code-switched, multilingual, and noisy inputs.

**Step 2 — Embedding Lookup**: Each discrete token index $t_i \in \{0, 1, \ldots, |V|-1\}$ is mapped to a dense continuous vector via a learned embedding matrix $\mathbf{W}_{\text{emb}} \in \mathbb{R}^{|V| \times d_{\text{model}}}$:

$$
E_{\text{text}} = \text{Embedding}\!\left(\text{BPE}(X_{\text{text}})\right) = \left[\mathbf{W}_{\text{emb}}[t_1],\; \mathbf{W}_{\text{emb}}[t_2],\; \ldots,\; \mathbf{W}_{\text{emb}}[t_L]\right] \in \mathbb{R}^{L \times d_{\text{model}}}
$$

where $L$ is the resulting sequence length after BPE tokenization and $d_{\text{model}}$ is the model hidden dimension.

**Key Properties**:
- Text tokens carry no intrinsic temporal, height, or width structure — this is handled by TM-RoPE (Section 4) by collapsing all three positional axes to identical values.

---

### 2.2 Audio Modality (Including Audio Extracted from Video)

**Raw Input**: Continuous-time audio waveform $X_{\text{audio}}(t)$ at arbitrary sample rate.

The audio processing pipeline comprises **four sequential stages**:

#### Stage 1 — Resampling

All raw waveforms are uniformly resampled to a canonical sample rate:

$$
f_s = 16{,}000 \text{ Hz} \quad (16 \text{ kHz})
$$

This ensures consistent spectral resolution regardless of the source encoding (e.g., 8 kHz telephony, 22.05 kHz podcast, 44.1 kHz music, 48 kHz video audio tracks).

#### Stage 2 — Mel-Spectrogram Extraction

The resampled waveform is transformed into a time-frequency representation via the **Short-Time Fourier Transform (STFT)** followed by a mel-scale filterbank projection:

- **Window size**: $w = 25 \text{ ms}$ (i.e., $25 \times 10^{-3} \times 16{,}000 = 400$ samples per window)
- **Hop length**: $h = 10 \text{ ms}$ (i.e., $10 \times 10^{-3} \times 16{,}000 = 160$ samples per hop)
- **Number of mel channels**: $M = 128$

$$
S_{\text{mel}} = \text{MelSpectrogram}(X_{\text{audio}}) \in \mathbb{R}^{128 \times T_{\text{mel}}}
$$

where the number of mel frames is:

$$
T_{\text{mel}} = \left\lfloor \frac{\text{(number of samples)} - 400}{160} \right\rfloor + 1
$$

Each mel frame corresponds to $10 \text{ ms}$ of the original audio signal. The 128-channel filterbank provides high spectral resolution across the perceptually relevant frequency range ($\sim 0$–$8$ kHz at $16$ kHz Nyquist).

#### Stage 3 — Conv2D Temporal Downsampling

The mel-spectrogram features (also called **filter bank features**) are passed through **sequential Conv2D downsampling blocks** that precede the attention layers of the AuT encoder:

$$
S_{\text{down}} = \text{Conv2D}_{\downarrow 8}(S_{\text{mel}})
$$

This achieves an **$8\times$ temporal reduction**. Since the original mel frame rate is $\frac{1}{10\text{ ms}} = 100 \text{ Hz}$:

$$
\text{Post-downsampling token rate} = \frac{100 \text{ Hz}}{8} = 12.5 \text{ Hz}
$$

**Physical interpretation**: Each downsampled frame encapsulates:

$$
\frac{1}{12.5 \text{ Hz}} = 80 \text{ ms}
$$

of the raw acoustic signal. This $80 \text{ ms}$ resolution becomes the **fundamental temporal quantum** for the entire multimodal system, governing TM-RoPE temporal ID increments and Talker generation rate.

#### Stage 4 — Audio Transformer (AuT) Encoder

The downsampled features are processed through the AuT encoder (detailed in Section 3) to produce dense audio representations:

$$
E_{\text{audio}} = \text{AuT}_{\text{Encoder}}\!\left(\text{Conv2D}_{\downarrow 8}(S_{\text{mel}})\right) \in \mathbb{R}^{T_{\text{audio}} \times d_{\text{model}}}
$$

where:

$$
T_{\text{audio}} = \left\lfloor \frac{T_{\text{mel}}}{8} \right\rfloor
$$

---

### 2.3 Vision Modality (Image and Video)

**Raw Input**: Static image $X_{\text{image}}$ or temporal video sequence $X_{\text{video}} = \{F_1, F_2, \ldots, F_N\}$.

#### Vision Encoder

The vision encoder is derived from **Qwen3-VL**, initialized from **SigLIP2-So400m**, comprising:

$$
\text{Vision Encoder Parameters} \approx 543\text{M}
$$

The encoder is trained on a mixture of image and video data, ensuring strong capabilities for both **static image understanding** and **temporal video comprehension**.

$$
E_{\text{vision}} = \text{VisionEncoder}(X_{\text{vision}}) \in \mathbb{R}^{T_{\text{vis}} \times d_{\text{model}}}
$$

#### Dynamic Frame Rate Sampling (Video)

For video inputs, frames are **not** extracted at a fixed uniform rate. Instead, a **dynamic frame rate sampling strategy** is employed with two objectives:

1. **Maximize spatiotemporal variance**: Select frames that carry maximal information content, avoiding redundant near-duplicate frames in static scenes while densely sampling during rapid motion.
2. **Synchronize with the acoustic token rate**: The visual frame sampling aligns with the $12.5 \text{ Hz}$ audio token rate to enable coherent temporal fusion across modalities.

#### Audio Extraction from Video

When a video contains an audio track, the audio channel is **extracted and processed independently** through the full audio pipeline (Section 2.2), including resampling to $16 \text{ kHz}$, mel-spectrogram extraction, Conv2D downsampling, and AuT encoding. The resulting audio and visual representations are then **temporally aligned** via TM-RoPE (Section 4).

---

### 2.4 Summary: Raw Input to Representation Mapping

| Modality | Raw Input | Preprocessing | Encoder | Output Token Rate | Output Shape |
|---|---|---|---|---|---|
| Text | Unicode string | BPE tokenization ($\|V\| = 151{,}643$) | Embedding lookup | 1 token per sub-word | $\mathbb{R}^{L \times d_{\text{model}}}$ |
| Audio | Waveform (any $f_s$) | Resample → 16 kHz, Mel-spec (128-ch, 25 ms win, 10 ms hop), Conv2D $\downarrow 8$ | AuT ($\approx 650$M) | $12.5 \text{ Hz}$ ($80$ ms/token) | $\mathbb{R}^{T_{\text{audio}} \times d_{\text{model}}}$ |
| Image | Pixel array | Encoder-specific preprocessing | SigLIP2-So400m ($\approx 543$M) | Spatial tokens per image | $\mathbb{R}^{T_{\text{vis}} \times d_{\text{model}}}$ |
| Video | Frame sequence + audio | Dynamic FPS sampling + audio extraction | SigLIP2-So400m (visual) + AuT (audio) | Synced to $12.5 \text{ Hz}$ | $\mathbb{R}^{T_{\text{vis}} \times d_{\text{model}}}$ + $\mathbb{R}^{T_{\text{audio}} \times d_{\text{model}}}$ |

---

## 3. Audio Transformer (AuT) Encoder — Detailed Architecture

### 3.1 Overview

The Audio Transformer (AuT) is an **attention-encoder-decoder** based autoregressive model, **trained from scratch** on a massive supervised audio corpus. Qwen3-Omni employs **only the encoder portion** of AuT as the audio encoder to obtain general-purpose audio representations.

### 3.2 Training Data Composition

| Data Category | Proportion | Description |
|---|---|---|
| Chinese & English pseudo-labeled ASR | $80\%$ | Automatic speech recognition with pseudo-labels |
| Other-language ASR | $10\%$ | Multilingual speech recognition coverage |
| Audio understanding | $10\%$ | Non-speech audio event detection, classification, captioning |

$$
\text{Total Training Data} = 20{,}000{,}000 \text{ hours of supervised audio}
$$

The mixed training objective — combining **speech recognition** with **audio understanding** — forces the encoder to learn representations that are simultaneously discriminative for linguistic content and robust to non-speech acoustic events, producing **general-purpose audio representations**.

### 3.3 Architectural Components

```
Raw Waveform (16 kHz)
        │
        ▼
┌─────────────────────┐
│  Mel-Spectrogram     │  128 channels, 25ms window, 10ms hop
│  Extraction          │  → 100 Hz frame rate
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  Sequential Conv2D   │  8× temporal downsampling
│  Downsampling Blocks │  → 12.5 Hz frame rate (80ms/frame)
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  Transformer Encoder │  ~0.6B parameters
│  (Flash Attention    │  Dynamic attention window: 1–8 seconds
│   with dynamic       │
│   window sizes)      │
└─────────┬───────────┘
          │
          ▼
    E_audio ∈ ℝ^{T_audio × d_model}
```

### 3.4 Conv2D Downsampling Blocks

The Conv2D blocks precede the attention layers and operate directly on the 2D mel-spectrogram tensor (frequency $\times$ time). The $8\times$ temporal reduction compresses:

$$
100 \text{ Hz (mel frame rate)} \xrightarrow{\text{Conv2D } \downarrow 8} 12.5 \text{ Hz (token rate)}
$$

This compression is critical for computational tractability: a 1-minute audio clip at $100 \text{ Hz}$ produces $6{,}000$ frames, which after $8\times$ downsampling yields only $750$ tokens — a manageable sequence length for transformer attention.

### 3.5 Dynamic Attention Window (Flash Attention)

AuT employs **flash attention** with **dynamic attention window sizes** spanning:

$$
w_{\text{attn}} \in [1\text{ s},\; 8\text{ s}]
$$

**Design rationale**: This intentionally balances two conflicting requirements:

| Requirement | Favored Window Size | Justification |
|---|---|---|
| Real-time prefill caching efficiency | Short windows ($\sim 1$s) | Enables low-latency incremental processing during streaming |
| Deep contextual offline modeling | Long windows ($\sim 8$s) | Captures prosodic contours, speaker turns, and extended acoustic context |

The dynamic mechanism adaptively selects window sizes during both training and inference, ensuring the encoder generalizes across streaming and offline operating regimes.

### 3.6 Key Specifications

| Specification | Value |
|---|---|
| Total parameters | $\approx 650\text{M}$ (encoder $\approx 600\text{M}$) |
| Training data | $20\text{M}$ hours |
| Input sample rate | $16 \text{ kHz}$ |
| Mel channels | $128$ |
| Window / Hop | $25 \text{ ms}$ / $10 \text{ ms}$ |
| Temporal downsampling factor | $8\times$ |
| Output token rate | $12.5 \text{ Hz}$ |
| Temporal resolution per token | $80 \text{ ms}$ |
| Attention type | Flash attention, dynamic window $[1\text{s}, 8\text{s}]$ |

---

## 4. Time-aligned Multimodal Rotary Position Embedding (TM-RoPE)

### 4.1 Problem Statement

Standard 1D Rotary Position Embedding (RoPE) assigns a single scalar position index to each token. When multiple modalities — each with distinct dimensionality (1D temporal for audio, 2D spatial for images, 3D spatiotemporal for video) — are concatenated into a single sequence, **catastrophic extrapolation degradation** occurs over extended multimodal contexts. Positional indices grow rapidly, exceeding the training-time range, and the rotary frequency spectrum cannot faithfully encode cross-modal temporal relationships.

### 4.2 Solution: 3D Factorized Rotary Embedding

TM-RoPE extends the Multimodal Rotary Position Embedding (M-RoPE) by incorporating **absolute temporal information**. The classical rotary embedding space is **orthogonally partitioned** into three factorized dimensions:

| Dimension | Symbol | Rotary Angles Allocated | Interpretation |
|---|---|---|---|
| Temporal | $\theta_T$ | $24$ | Absolute time position |
| Height | $\theta_H$ | $20$ | Spatial row coordinate |
| Width | $\theta_W$ | $20$ | Spatial column coordinate |

$$
\text{Total rotary angles} = 24 + 20 + 20 = 64
$$

For a token at position $(t, h, w)$, the rotary embedding is applied as the **concatenation of three independent rotary transformations**, each operating on its allocated subset of the hidden dimension:

$$
\text{RoPE}_{\text{TM}}(x, t, h, w) = \text{RoPE}_T(x_{[1:24]}, t) \;\|\; \text{RoPE}_H(x_{[25:44]}, h) \;\|\; \text{RoPE}_W(x_{[45:64]}, w)
$$

where $\|$ denotes concatenation along the rotary angle dimension.

### 4.3 Modality-Specific Positional ID Assignment Rules

#### 4.3.1 Text

Text is inherently one-dimensional. All three positional components receive **identical** IDs:

$$
\forall\; \text{token}_i \in E_{\text{text}}: \quad \text{ID}_T = \text{ID}_H = \text{ID}_W = \text{ID}_{\text{base}} + i
$$

**Effect**: TM-RoPE mathematically **collapses to standard 1D RoPE** for text, since all three rotary sub-spaces rotate identically. This ensures backward compatibility with text-only pre-training.

#### 4.3.2 Audio

Audio tokens are one-dimensional in time but require **absolute temporal anchoring** for cross-modal alignment:

$$
\forall\; \text{frame}_t \in E_{\text{audio}}: \quad \text{ID}_T = \text{ID}_{\text{base}} + \left\lfloor \frac{t}{80\text{ ms}} \right\rfloor, \quad \text{ID}_H = \text{ID}_W = \text{ID}_T
$$

The temporal ID increments **exactly once every $80 \text{ ms}$**, matching the AuT encoder's output token rate of $12.5 \text{ Hz}$. Height and width components share the temporal ID (audio has no spatial structure).

#### 4.3.3 Vision — Static Images

For still images, there is **no temporal progression**; the temporal ID remains constant while spatial tokens are distributed across row ($r$) and column ($c$) coordinates:

$$
\text{ID}_T = \text{constant}, \quad \text{ID}_H = r, \quad \text{ID}_W = c
$$

This encodes the 2D spatial layout of image patches while fixing the temporal axis.

#### 4.3.4 Vision — Video Frames

For video, temporal IDs are **monotonically increasing**, dynamically adjusted according to raw timestamps to maintain strict $80 \text{ ms}$ resolution per ID:

$$
\text{ID}_T = f(t_{\text{absolute}}), \quad \text{ID}_H = r, \quad \text{ID}_W = c
$$

where $f(\cdot)$ maps the absolute timestamp to the corresponding $80 \text{ ms}$-quantized temporal ID. Within each frame, spatial tokens are distributed across $(r, c)$ as with static images.

### 4.4 Cross-Modal Contiguity Enforcement

To prevent **positional ID collisions** between tokens from different modalities within the concatenated sequence, a strict contiguity rule is enforced:

$$
\text{ID}_{\text{base}}^{(m)} = 1 + \max\!\left(\text{ID}_{\text{assigned}}^{(m-1)}\right)
$$

Each sequential modality block initializes its indices at $1 + \max(\text{previous modality's assigned IDs})$. This guarantees:

1. **No overlapping position IDs** across modalities.
2. **Monotonic global ordering** within the full multimodal sequence.
3. Audio and video representations are **directly aligned using their temporal IDs**, bypassing fixed chunking and facilitating **arbitrary streaming input durations**.

### 4.5 Design Rationale for Rotary Angle Allocation

The original M-RoPE formulation allocated the initial $16$ rotary angles to temporal dependencies, corresponding to **higher frequencies** with stronger oscillatory patterns. While effective for fine-grained local temporal variation, this design suffered when modeling **long-range temporal dependencies** across extended multimodal contexts.

TM-RoPE addresses this by:
- Expanding temporal allocation to **$24$ angles** (from $16$), providing a richer frequency spectrum for temporal encoding.
- Incorporating **absolute temporal information** rather than relying solely on relative positional differences, preventing degradation during sequence extrapolation.

---

## 5. Thinker Module — Cognitive Reasoning Engine

### 5.1 Architecture

The Thinker is the **primary cognitive reasoning backbone** of Qwen3-Omni, implemented as a **Mixture-of-Experts (MoE) Transformer**:

$$
\text{Thinker}: \quad 30\text{B total parameters}, \quad 3\text{B active parameters per token}
$$

The MoE architecture ensures that only a small fraction ($\frac{3\text{B}}{30\text{B}} = 10\%$) of parameters are activated for any given token, providing:
- **High parameter capacity** for knowledge storage and multimodal reasoning.
- **Low per-token FLOP cost** for efficient inference.
- **Reduced KV cache I/O** compared to equivalently capable dense models, critically important for long multimodal sequences.

### 5.2 Input Processing

The Thinker ingests the **temporally aligned multimodal representations** from all encoders:

$$
H_{\text{Thinker}}^{(N)} = \text{MoE}_{\text{Thinker}}\!\left(E_{\text{text}}^{(N)},\; E_{\text{audio}}^{(N)},\; E_{\text{vision}}^{(N)}\right)
$$

All representations carry TM-RoPE positional encodings, enabling the attention mechanism to correctly attend across modality boundaries with proper temporal and spatial awareness.

### 5.3 Outputs

The Thinker simultaneously produces:

1. **Textual response tokens** $Y_{\text{text}}^{(N)}$: The autoregressive text generation output.
2. **High-dimensional multimodal features** $H_{\text{Thinker}}^{(N)}$: Rich, contextualized representations that are forwarded to the Talker for speech synthesis conditioning.

### 5.4 Independent System Prompt

The Thinker receives its own **dedicated system prompt**, controlling:
- Response style (formal, conversational, concise, detailed)
- Reasoning depth
- Language preferences
- Content policy adherence

This is independent of the Talker's system prompt, enabling **fine-grained control** over cognitive reasoning separately from acoustic generation characteristics.

---

## 6. Talker Module — Streaming Speech Generation Engine

### 6.1 Architecture

The Talker is a **lightweight MoE Transformer** specifically designed for streaming acoustic token generation:

$$
\text{Talker}: \quad 3\text{B total parameters}, \quad 0.3\text{B active parameters per token}
$$

The dramatically smaller active parameter count ($0.3\text{B}$) compared to the Thinker ($3\text{B}$) reflects the Talker's narrower task: converting contextualized multimodal features into a sequence of codec tokens, rather than performing open-ended multimodal reasoning.

### 6.2 Conditioning (Qwen3-Omni Design)

**Critical architectural change**: In Qwen3-Omni, the Talker **no longer consumes the Thinker's high-level text representations**. It conditions **only on audio and visual multimodal features**:

$$
H_{\text{Talker}}^{(N-1)} = \text{MoE}_{\text{Talker}}\!\left(H_{\text{Thinker}}^{(N-1)},\; E_{\text{audio\_ctx}},\; E_{\text{vision\_ctx}}\right)
$$

where $H_{\text{Thinker}}^{(N-1)}$ refers to the high-dimensional multimodal features (not textual token representations) from the previous chunk, and $E_{\text{audio\_ctx}}$, $E_{\text{vision\_ctx}}$ are the audio and visual context embeddings.

**Textual content** is supplied to the Talker via **discrete tokens** through controlled preprocessing, not via the Thinker's internal representations. This decoupling enables:
- External intervention (RAG, function calling, safety filters) on the Thinker's textual output before it reaches the Talker.
- Supplying modified or filtered text to the Talker for streaming synthesis.
- Independent control of response content (Thinker) and speech characteristics (Talker).

### 6.3 Independent System Prompt

The Talker has its own dedicated system prompt controlling:
- Voice identity / timbre
- Speaking rate
- Prosodic style
- Emotional expression
- Language/accent for synthesis

### 6.4 Output

At each generation step ($12.5 \text{ Hz}$ frame rate), the Talker produces the **primary codebook token** $C_0^{(t)}$ for the current frame:

$$
C_0^{(t)} = \text{LinearHead}\!\left(H_{\text{Talker}}^{(t)}\right)
$$

This token is immediately forwarded to the MTP module (Section 7) for residual codebook completion.

---

## 7. Multi-Codebook Speech Synthesis — MTP Module

### 7.1 Codec Framework: Residual Vector Quantization (RVQ)

The speech synthesis pipeline employs **Residual Vector Quantization (RVQ)**, where each $80 \text{ ms}$ audio frame is represented by a **stack of $K+1$ codebook tokens** $\{C_0, C_1, \ldots, C_K\}$:

- $C_0$: **Primary codebook** — captures the coarse spectral envelope and fundamental acoustic structure.
- $C_1, \ldots, C_K$: **Residual codebooks** — progressively encode finer acoustic details, harmonics, and texture that the primary codebook cannot represent.

Each successive codebook quantizes the **residual error** from the previous codebooks, hence the name "residual" vector quantization.

### 7.2 Multi-Token Prediction (MTP) Module Architecture

The MTP module is an **ultra-lightweight, fixed-step autoregressive Dense Transformer**:

$$
\text{MTP Module}: \quad \approx 80\text{M parameters}
$$

**Design properties**:
- **Dense** (not MoE) — the module is small enough that MoE sparsity is unnecessary.
- **Fixed-step autoregressive** — the number of residual codebooks $K$ is fixed, so the autoregressive generation runs for exactly $K$ steps per frame.
- **Left-context-only attention** — strict causal masking ensures streaming compatibility; the MTP module attends exclusively to previously generated tokens, never to future frames.
- **Low memory bandwidth requirements** — enables efficient batch processing of high-throughput requests.
- **Fixed KV cache** — the fixed-step nature allows pre-allocated, constant-size KV cache for acceleration.

### 7.3 Generation Algorithm

For each temporal generation step $t$ at the $12.5 \text{ Hz}$ frame rate:

**Step 1 — Primary Codebook (from Talker)**:

$$
C_0^{(t)} = \text{LinearHead}\!\left(H_{\text{Talker}}^{(t)}\right)
$$

**Step 2 — Residual Codebooks (from MTP)**:

$$
C_{1 \ldots K}^{(t)} = \text{MTP}\!\left(C_0^{(t)},\; \text{Context}_{<t}\right)
$$

The MTP module takes the primary codebook token $C_0^{(t)}$ as the initial input and autoregressively generates $C_1^{(t)}, C_2^{(t)}, \ldots, C_K^{(t)}$, each conditioned on all previously generated codebooks for the current frame and the left context from prior frames.

**Step 3 — Forward to Code2Wav**: The complete codebook stack $\{C_0^{(t)}, C_1^{(t)}, \ldots, C_K^{(t)}\}$ is immediately forwarded to the Code2Wav ConvNet for waveform synthesis.

---

## 8. Code2Wav — Causal Convolutional Waveform Decoder

### 8.1 Architecture

$$
\text{Code2Wav}: \quad \approx 200\text{M parameters}, \quad \text{Causal ConvNet}
$$

The Code2Wav module is a **lightweight causal convolutional neural network** functioning as a **neural vocoder**. It replaces the more complex DiT-based (Diffusion Transformer) vocoder used in Qwen2.5-Omni.

### 8.2 Operation

For each frame $t$, the Code2Wav ConvNet takes the complete multi-codebook sequence and **instantaneously synthesizes** the corresponding continuous waveform:

$$
\hat{W}^{(t)} = \text{Code2Wav}_{\text{ConvNet}}\!\left(C_{0 \ldots K}^{(t)}\right)
$$

Each output $\hat{W}^{(t)}$ corresponds to approximately $80 \text{ ms}$ of audio waveform (matching the $12.5 \text{ Hz}$ frame rate).

### 8.3 Design Advantages

| Property | Benefit |
|---|---|
| **Causal convolution** | Enables streaming — only depends on current and past frames, never future |
| **ConvNet architecture** | Extensive hardware acceleration support (CUDA, TensorRT, etc.) across diverse inference platforms |
| **Lightweight** ($200\text{M}$) | Low computational FLOPs, low latency per frame |
| **Batched inference** | Efficient for high-concurrency scenarios |
| **No iterative diffusion** | Single forward pass per frame (vs. multiple denoising steps in DiT), dramatically reducing inference latency |
| **Left-context-only** | Matches the causal streaming constraint — no need to wait for future context |

### 8.4 Comparison with DiT-based Vocoder (Qwen2.5-Omni)

| Aspect | Qwen2.5-Omni (DiT) | Qwen3-Omni (ConvNet) |
|---|---|---|
| Synthesis trigger | Must wait for sufficient **block-context** from Talker | Immediate output after **each Talker token** |
| Computational cost | Multiple denoising steps per block | Single forward pass per frame |
| Latency | Higher first-packet latency | Significantly reduced first-packet latency |
| Hardware efficiency | Requires attention computation | Pure convolution — highly optimized |
| Audio fidelity | High | **Superior** (per the paper's claims) |

---

## 9. Asynchronous Chunked Prefilling and Streaming Pipeline

### 9.1 Chunk-based Processing

Both the audio and vision encoders are capable of outputting representations **along the temporal dimension in chunks**, enabling incremental processing without waiting for the entire input to complete.

### 9.2 Asynchronous Thinker-Talker Execution

The key latency optimization is the **asynchronous interleaving** of Thinker and Talker prefilling:

```
Timeline:
─────────────────────────────────────────────────────────
Thinker:  [Prefill Chunk N  ] [Prefill Chunk N+1] [Prefill Chunk N+2] ...
Talker:         [Prefill Chunk N-1] [Prefill Chunk N  ] [Prefill Chunk N+1] ...
MTP:                 [Gen Frame] [Gen Frame] [Gen Frame] ...
Code2Wav:                 [Render] [Render] [Render] ...
Audio Out:                   🔊      🔊      🔊    ...
─────────────────────────────────────────────────────────
```

**Formal algorithm for chunk $N$:**

**Step 1 — Thinker Prefill (Chunk $N$)**:

$$
H_{\text{Thinker}}^{(N)} = \text{MoE}_{\text{Thinker}}\!\left(E_{\text{text}}^{(N)},\; E_{\text{audio}}^{(N)},\; E_{\text{vision}}^{(N)}\right)
$$

The Thinker ingests chunk $N$'s multimodal representations, producing text tokens $Y_{\text{text}}^{(N)}$ and high-dimensional features.

**Step 2 — Talker Prefill (Chunk $N-1$) [Asynchronous]**:

$$
H_{\text{Talker}}^{(N-1)} = \text{MoE}_{\text{Talker}}\!\left(H_{\text{Thinker}}^{(N-1)},\; E_{\text{audio\_ctx}},\; E_{\text{vision\_ctx}}\right)
$$

**Simultaneously** while the Thinker processes chunk $N$, the Talker prefills chunk $N-1$ using the Thinker's previously computed features. This **overlapped execution** significantly reduces the effective Time-To-First-Token (TTFT) for both modules.

**Step 3 — MTP + Code2Wav (Streaming)**:

Once the Talker generates its first token for chunk $N-1$:
1. MTP immediately predicts the residual codebooks for that frame.
2. Code2Wav immediately renders the waveform.
3. The waveform is yielded to the streaming audio output buffer.

### 9.3 Why MoE Enhances Concurrency

The MoE architecture provides specific advantages for streaming multimodal inference:

| MoE Advantage | Mechanism |
|---|---|
| **Reduced KV cache I/O** | Only active expert parameters contribute to KV computations, reducing memory bandwidth during long-sequence processing |
| **Higher tokens per second (TPS)** | Lower per-token FLOPs → faster generation |
| **Better concurrency scaling** | Prefill latency and TTFT remain **largely unaffected** under high concurrency (see latency analysis below) |

---

## 10. End-to-End First-Packet Latency Analysis

### 10.1 Latency Component Decomposition

The total first-packet latency is the **sequential sum** of all pipeline stages (due to their data dependencies):

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{preprocessing}} + \mathcal{L}_{\text{Thinker-TTFT}} + \mathcal{L}_{\text{Talker-TTFT}} + \mathcal{L}_{\text{MTP}} + \mathcal{L}_{\text{Code2Wav}}
$$

### 10.2 Measured Latencies (vLLM Framework, torch.compile + CUDA Graph)

| Component | 1 Concurrent | 4 Concurrent | 6 Concurrent |
|---|---|---|---|
| Thinker-Talker Tail Packet Preprocessing (Audio/Video) | $72 / 160$ ms | $94 / 180$ ms | $100 / 200$ ms |
| Thinker TTFT (Audio/Video) | $88 / 160$ ms | $468 / 866$ ms | $673 / 1{,}330$ ms |
| Talker TTFT (Audio/Video) | $57 / 210$ ms | $145 / 450$ ms | $376 / 734$ ms |
| MTP Module (per token) | $14$ ms | $16$ ms | $18$ ms |
| Codec Decoder (per code) | $3$ ms | $5$ ms | $5$ ms |
| **Overall First-Packet Latency (Audio/Video)** | **$234 / 547$ ms** | **$728 / 1{,}517$ ms** | **$1{,}172 / 2{,}284$ ms** |

### 10.3 Generation Real-Time Factor (RTF)

After the first packet is emitted and the model enters **streaming audio synthesis mode**, the Talker operates at $12.5 \text{ Hz}$ — requiring only **one token to synthesize $80 \text{ ms}$ of audio**. The Generation RTF is:

$$
\text{RTF} = \frac{\mathcal{L}_{\text{Thinker-gen}} + \mathcal{L}_{\text{Talker-gen}} + \mathcal{L}_{\text{MTP}} + \mathcal{L}_{\text{Code2Wav}}}{80 \text{ ms}}
$$

| Concurrency | Thinker TPS | Talker TPS | Generation RTF |
|---|---|---|---|
| 1 | 75 tokens/s | 140 tokens/s | $0.47$ |
| 4 | 63 tokens/s | 125 tokens/s | $0.56$ |
| 6 | 53 tokens/s | 110 tokens/s | $0.66$ |

$$
\text{RTF} < 1 \quad \forall \text{ tested concurrency levels}
$$

**Interpretation**: RTF $< 1$ guarantees that the system generates audio **faster than real time**, ensuring users receive **continuously streaming audio** without interruptions or buffering gaps.

---

## 11. Complete End-to-End Data Flow Diagram

```
╔══════════════════════════════════════════════════════════════════════════════════════╗
║                         RAW MULTIMODAL INPUTS                                       ║
╠═══════════════╦═══════════════════╦══════════════════╦══════════════════════════════╣
║  Text String  ║  Audio Waveform   ║  Static Image    ║  Video (Frames + Audio)      ║
╚═══════╤═══════╩═════════╤═════════╩════════╤═════════╩══════════════╤═══════════════╝
        │                 │                  │                        │
        ▼                 ▼                  │              ┌────────┴────────┐
   ┌─────────┐    ┌──────────────┐           │              │                 │
   │  BPE    │    │ Resample     │           │         ┌────▼─────┐    ┌──────▼──────┐
   │Tokenizer│    │  → 16 kHz    │           │         │ Dynamic  │    │ Extract     │
   │|V|=     │    └──────┬───────┘           │         │ FPS      │    │ Audio Track │
   │151,643  │           │                   │         │ Sampling │    └──────┬──────┘
   └────┬────┘           ▼                   │         └────┬─────┘          │
        │         ┌──────────────┐           │              │           (→ Audio Pipeline)
        ▼         │ Mel-Spectro  │           │              │                │
   ┌─────────┐   │ 128ch, 25ms  │           │              │                │
   │Embedding│   │ win, 10ms    │           │              │                │
   │ Lookup  │   │ hop → 100Hz  │           │              │                │
   └────┬────┘   └──────┬───────┘           │              │                │
        │               │                   │              │                │
        │               ▼                   │              │                │
        │        ┌──────────────┐           │              │                │
        │        │ Conv2D ↓8    │           │              │                │
        │        │ → 12.5 Hz   │           │              │                │
        │        └──────┬───────┘           │              │                │
        │               │                   │              │                │
        │               ▼                   ▼              ▼                │
        │        ┌──────────────┐   ┌──────────────┐                       │
        │        │  AuT Encoder │   │SigLIP2-So400m│                       │
        │        │  (~650M)     │   │Vision Encoder│                       │
        │        │  Flash Attn  │   │  (~543M)     │                       │
        │        │  1-8s window │   └──────┬───────┘                       │
        │        └──────┬───────┘          │                               │
        │               │                  │                               │
        ▼               ▼                  ▼                               ▼
   ┌─────────┐   ┌──────────┐      ┌──────────┐                    ┌──────────┐
   │ E_text  │   │ E_audio  │      │ E_vision │                    │ E_audio  │
   │         │   │ 12.5Hz   │      │          │                    │ (video)  │
   └────┬────┘   └────┬─────┘      └────┬─────┘                    └────┬─────┘
        │              │                 │                               │
        ╚══════════════╩═════════════════╩═══════════════════════════════╝
                                         │
                                         ▼
                              ┌──────────────────────┐
                              │     TM-RoPE          │
                              │  Positional Encoding │
                              │  (24T + 20H + 20W)   │
                              └──────────┬───────────┘
                                         │
                                         ▼
                    ╔════════════════════════════════════════╗
                    ║         THINKER (MoE 30B-A3B)         ║
                    ║  Multimodal Reasoning & Text Gen      ║
                    ╠═══════════════╦════════════════════════╣
                    ║ Y_text (text) ║ H_Thinker (features)  ║
                    ╚═══════╤═══════╩═══════════╤════════════╝
                            │                   │
                   [External│Intervention]      │
                   [RAG/Safety/FC]              │
                            │                   │
                            ▼                   ▼
                    ┌───────────┐    ╔══════════════════════════╗
                    │ Text Out  │    ║  TALKER (MoE 3B-A0.3B)  ║
                    │ (stream)  │    ║  Conditions on:          ║
                    └───────────┘    ║  • Multimodal features   ║
                                    ║  • Audio/Visual context   ║
                                    ║  • Text (via preprocess)  ║
                                    ╠══════════════════════════╣
                                    ║  Output: C_0 per frame   ║
                                    ╚═══════════╤══════════════╝
                                                │
                                                ▼
                                    ╔══════════════════════════╗
                                    ║  MTP Module (80M Dense)  ║
                                    ║  Fixed-step AR generation║
                                    ║  Left-context only       ║
                                    ╠══════════════════════════╣
                                    ║  Output: C_1...C_K       ║
                                    ╚═══════════╤══════════════╝
                                                │
                                                ▼
                                    ╔══════════════════════════╗
                                    ║  Code2Wav ConvNet (200M) ║
                                    ║  Causal convolutional    ║
                                    ║  Neural vocoder          ║
                                    ╠══════════════════════════╣
                                    ║  Output: Ŵ(t) waveform  ║
                                    ╚═══════════╤══════════════╝
                                                │
                                                ▼
                                        🔊 Streaming Audio
                                     (80ms per frame, RTF<1)
```

---

## 12. Consolidated Algorithmic Pipeline

### Algorithm: Complete End-to-End Inference

**Input**: Raw multimodal input $\{X_{\text{text}}, X_{\text{audio}}, X_{\text{image}}, X_{\text{video}}\}$ (any subset may be present)

**Output**: Streaming text tokens $Y_{\text{text}}$ and streaming audio waveform $\hat{W}$

---

**Phase 1: Perceivation (Parallel Encoding)**

$$
E_{\text{text}} = \text{Embedding}\!\left(\text{BPE}(X_{\text{text}})\right)
$$

$$
S_{\text{mel}} = \text{MelSpectrogram}_{128\text{ch, 25ms, 10ms}}(X_{\text{audio}} \downarrow 16\text{kHz})
$$

$$
E_{\text{audio}} = \text{AuT}_{\text{Encoder}}\!\left(\text{Conv2D}_{\downarrow 8}(S_{\text{mel}})\right) \quad \text{at } 12.5 \text{ Hz}
$$

$$
E_{\text{vision}} = \text{VisionEncoder}_{\text{SigLIP2}}\!\left(X_{\text{vision}}\right)
$$

---

**Phase 2: TM-RoPE Position Assignment**

Assign $(\text{ID}_T, \text{ID}_H, \text{ID}_W)$ per token according to modality rules (Section 4.3), enforcing cross-modal contiguity.

---

**Phase 3: Asynchronous Thinker-Talker Streaming Loop**

$\textbf{for}$ each chunk $N = 1, 2, \ldots$ $\textbf{do}$:

$\quad$ **[Thinker — Chunk $N$]** (asynchronous with Talker Chunk $N-1$):

$$
H_{\text{Thinker}}^{(N)}, \; Y_{\text{text}}^{(N)} = \text{MoE}_{\text{Thinker}}\!\left(E_{\text{text}}^{(N)},\; E_{\text{audio}}^{(N)},\; E_{\text{vision}}^{(N)}\right)
$$

$\quad$ $\textbf{yield}$ $Y_{\text{text}}^{(N)}$ to text output stream

$\quad$ **[Talker — Chunk $N-1$]** (asynchronous with Thinker Chunk $N$):

$$
H_{\text{Talker}}^{(N-1)} = \text{MoE}_{\text{Talker}}\!\left(H_{\text{Thinker}}^{(N-1)},\; E_{\text{audio\_ctx}},\; E_{\text{vision\_ctx}}\right)
$$

$\quad$ $\textbf{for}$ each frame $t$ at $12.5 \text{ Hz}$ $\textbf{do}$:

$\quad\quad$ **[Primary Codebook]**:

$$
C_0^{(t)} = \text{LinearHead}\!\left(H_{\text{Talker}}^{(t)}\right)
$$

$\quad\quad$ **[Residual Codebooks — MTP]**:

$$
C_{1 \ldots K}^{(t)} = \text{MTP}\!\left(C_0^{(t)},\; \text{Context}_{<t}\right)
$$

$\quad\quad$ **[Waveform Rendering — Code2Wav]**:

$$
\hat{W}^{(t)} = \text{Code2Wav}_{\text{ConvNet}}\!\left(C_{0 \ldots K}^{(t)}\right)
$$

$\quad\quad$ $\textbf{yield}$ $\hat{W}^{(t)}$ to streaming audio buffer

$\quad$ $\textbf{end for}$

$\textbf{end for}$

---

**Theoretical first-packet latency (cold start)**: $234 \text{ ms}$ (audio-only) / $547 \text{ ms}$ (audio+video) at 1 concurrency.

---

## 13. Key Design Invariants and Constraints

| Invariant | Value / Rule | Consequence |
|---|---|---|
| Fundamental temporal quantum | $80 \text{ ms}$ | Governs AuT output rate, TM-RoPE temporal ID increment, Talker generation rate, and waveform frame duration |
| Audio token rate | $12.5 \text{ Hz}$ | $= \frac{1}{80\text{ ms}}$; one Talker token → one $80\text{ ms}$ audio frame |
| Mel-spectrogram frame rate | $100 \text{ Hz}$ | $= \frac{1}{10\text{ ms hop}}$; before $8\times$ Conv2D downsampling |
| Temporal downsampling | $8\times$ | $100 \text{ Hz} \rightarrow 12.5 \text{ Hz}$ |
| TM-RoPE dimension split | $24 + 20 + 20 = 64$ | Temporal, Height, Width |
| BPE vocabulary | $151{,}643$ | Regular tokens (byte-level, zero OOV) |
| RTF constraint | $< 1$ always | Audio generated faster than real-time at all tested concurrencies |
| Causal constraint | Left-context only | MTP, Code2Wav, and Talker all enforce strict causal attention for streaming |
| Thinker-Talker text decoupling | Talker receives no text representations from Thinker | Enables independent intervention, separate system prompts, modular control |



# End-to-End Architectural Decomposition of Qwen3-Omni

---

## Table of Contents

1. [System-Level Overview](#1-system-level-overview)
2. [Multimodal Input Processing and Perceivation (Raw Inputs)](#2-multimodal-input-processing-and-perceivation)
3. [Audio Transformer (AuT) Encoder](#3-audio-transformer-aut-encoder)
4. [Vision Encoder](#4-vision-encoder)
5. [Time-aligned Multimodal Rotary Position Embedding (TM-RoPE)](#5-time-aligned-multimodal-rotary-position-embedding-tm-rope)
6. [Thinker Module — Cognitive Reasoning Engine](#6-thinker-module)
7. [Talker Module — Streaming Acoustic Generator](#7-talker-module)
8. [Multi-Token Prediction (MTP) Module](#8-multi-token-prediction-mtp-module)
9. [Code2Wav ConvNet Decoder](#9-code2wav-convnet-decoder)
10. [Asynchronous Chunked Prefilling and Streaming Pipeline](#10-asynchronous-chunked-prefilling-and-streaming-pipeline)
11. [End-to-End Algorithmic Pipeline (Unified Formal Description)](#11-end-to-end-algorithmic-pipeline)
12. [Latency Analysis and Concurrency Characteristics](#12-latency-analysis-and-concurrency-characteristics)
13. [Complete Module Summary Table](#13-complete-module-summary-table)

---

## 1. System-Level Overview

Qwen3-Omni implements a **natively multimodal Thinker-Talker Mixture-of-Experts (MoE) architecture** designed to process and synthesize four modalities — **text, image, video, and audio** — within a single cohesive model. The fundamental architectural principle is the **decoupling of high-level cognitive reasoning (Thinker) from streaming acoustic generation (Talker)**, while maintaining shared representational spaces that enable end-to-end training and unified inference.

### 1.1 Core Design Principles

| Principle | Implementation |
|---|---|
| **Single-Model Cohesion** | Talker directly ingests high-dimensional multimodal features from Thinker; shared conversational history; end-to-end trainable |
| **Cognitive-Acoustic Decoupling** | Thinker produces text tokens + high-level features; Talker conditions on multimodal features (not text representations) for speech |
| **Textual Representation Independence** | Thinker and Talker use **distinct system prompts** — independently controlling response style and audio style |
| **Streaming-First Design** | Every module (AuT, Thinker, Talker, MTP, Code2Wav) supports temporal chunking and causal (left-context-only) operation |
| **MoE Scalability** | Both Thinker and Talker use MoE to maximize throughput, reduce KV-cache I/O, and sustain low latency under high concurrency |

### 1.2 Architectural Signal Flow (High-Level)

```
Raw Inputs
├── Text ──────────────→ BPE Tokenizer ──→ Embedding ──────────────────────┐
├── Audio ─────────────→ 16kHz Resample → Mel-Spec → Conv2D↓8 → AuT Enc ─┤
├── Image ─────────────→ Vision Encoder (SigLIP2-So400m) ─────────────────┤
└── Video ─────────────→ Dynamic FPS Sampling → Vision Encoder ───────────┤
                         Video Audio Track → AuT Encoder ─────────────────┤
                                                                          │
                              ┌────────────────────────────────────────────┘
                              ▼
                     TM-RoPE Position Assignment
                              │
                              ▼
               ┌──────────────────────────┐
               │   THINKER (30B-A3B MoE)  │──→ Text Response Tokens Y_text
               │   Cognitive Reasoning    │──→ High-Dimensional Features H_Thinker
               └──────────┬───────────────┘
                          │ (Multimodal features: audio + vision only)
                          │ (NOT text representations)
                          ▼
               ┌──────────────────────────┐
               │   TALKER (3B-A0.3B MoE)  │──→ Primary Codebook Token C_0
               │   Acoustic Generation    │
               └──────────┬───────────────┘
                          │
                          ▼
               ┌──────────────────────────┐
               │   MTP (80M Dense Trans.) │──→ Residual Codebook Tokens C_{1..K}
               │   Multi-Token Prediction │
               └──────────┬───────────────┘
                          │
                          ▼
               ┌──────────────────────────┐
               │   Code2Wav (200M ConvNet)│──→ Waveform Ŵ(t)
               │   Neural Vocoder        │
               └──────────┬───────────────┘
                          │
                          ▼
                  Streaming Audio Output
```

### 1.3 Key Architectural Changes from Qwen2.5-Omni

1. **MoE Adoption**: Both Thinker and Talker migrate from dense transformers to MoE transformers for superior concurrency and inference speed.
2. **Talker Conditioning Decoupling**: Talker **no longer consumes** Thinker's high-level text representations. It conditions **only** on audio and visual multimodal features. Rationale:
   - For textual content, discrete tokens and embeddings are **information-equivalent**.
   - Multimodal conditioning is necessary for audio-video-coordinated speech generation (e.g., preserving prosody/timbre in speech translation).
   - This decoupling permits external modules (RAG, function calling, safety filters) to intervene on Thinker's textual output before optionally supplying modified text to the Talker via controlled preprocessing.
3. **Multi-Codebook Autoregressive Scheme**: Talker generates one codec frame per step; a dedicated MTP module produces the remaining residual codebooks.
4. **Lightweight Causal ConvNet Vocoder**: Code2Wav replaces block-context DiT-based vocoders, enabling immediate per-frame waveform rendering.

---

## 2. Multimodal Input Processing and Perceivation

The perceivation pipeline strictly harmonizes heterogeneous raw data streams into an aligned, high-dimensional representational space before routing to the Thinker's MoE layers.

### 2.1 Text Modality

**Input**: Raw text string $X_{\text{text}}$

**Processing Pipeline**:

1. **Tokenization**: Byte-level Byte-Pair Encoding (BPE) with vocabulary size $V = 151{,}643$ regular tokens (Qwen tokenizer).
2. **Embedding**: Discrete token indices are mapped to dense continuous embedding vectors.

$$
E_{\text{text}} = \text{Embedding}\bigl(\text{BPE}(X_{\text{text}})\bigr) \in \mathbb{R}^{L_{\text{text}} \times d_{\text{model}}}
$$

where $L_{\text{text}}$ is the token sequence length and $d_{\text{model}}$ is the hidden dimension of the Thinker.

**Key Details**:
- Byte-level BPE operates on raw bytes, ensuring lossless coverage of arbitrary Unicode input without unknown-token fallback.
- The vocabulary of $151{,}643$ tokens is shared across the Qwen model family.

### 2.2 Audio Modality

**Input**: Raw audio waveform $X_{\text{audio}}$ (standalone audio or audio track extracted from video)

**Processing Pipeline**:

| Step | Operation | Output |
|------|-----------|--------|
| 1 | Resample to $16\text{ kHz}$ | Uniform sample rate waveform |
| 2 | Extract 128-channel mel-spectrogram ($25\text{ ms}$ window, $10\text{ ms}$ hop) | $S_{\text{mel}} \in \mathbb{R}^{128 \times T_{\text{frames}}}$ |
| 3 | Conv2D downsampling ($8\times$ temporal reduction) | Compressed feature sequence |
| 4 | Audio Transformer (AuT) Encoder | $E_{\text{audio}} \in \mathbb{R}^{L_{\text{audio}} \times d_{\text{model}}}$ |

**Formal Computation**:

$$
S_{\text{mel}} = \text{MelSpectrogram}_{128,\,25\text{ms},\,10\text{ms}}(X_{\text{audio}})
$$

$$
E_{\text{audio}} = \text{AuT}_{\text{Encoder}}\bigl(\text{Conv2D}_{\downarrow 8}(S_{\text{mel}})\bigr)
$$

**Derived Token Rate Calculation**:

The raw mel-spectrogram at $10\text{ ms}$ hop produces $100$ frames per second. After $8\times$ temporal downsampling:

$$
\text{Token Rate} = \frac{100 \text{ frames/s}}{8} = 12.5 \text{ Hz}
$$

Each discrete audio token therefore encapsulates exactly:

$$
\text{Temporal Coverage per Token} = \frac{1}{12.5} = 80 \text{ ms}
$$

### 2.3 Vision Modality (Image and Video)

**Input**: Image $X_{\text{image}}$ or video $X_{\text{video}}$ (visual frames only; audio track handled separately by §2.2)

**Processing Pipeline**:

1. **Frame Extraction (Video Only)**: Dynamic frame rate sampling strategy that:
   - Preserves maximum spatiotemporal variance across the video.
   - Aligns with the $12.5\text{ Hz}$ acoustic token rate for cross-modal temporal synchronization.
2. **Visual Encoding**: Process through the vision encoder initialized from **SigLIP2-So400m** ($\sim 543\text{M}$ parameters), derived from Qwen3-VL.

$$
E_{\text{vision}} = \text{VisionEncoder}_{543\text{M}}(X_{\text{vision}})
$$

**Key Details**:
- The vision encoder is trained on a mixture of image and video data, ensuring strong performance on both static image understanding and dynamic video comprehension.
- For video inputs, the audio track is extracted and processed independently through the AuT pipeline (§2.2), then temporally aligned with visual representations via TM-RoPE (§5).

### 2.4 Unified Representation Space

After perceivation, all modality embeddings share a common dimensionality $d_{\text{model}}$ and are concatenated along the sequence dimension with TM-RoPE positional identifiers assigned per-token:

$$
E_{\text{multimodal}} = \text{Concat}\bigl[E_{\text{text}},\; E_{\text{audio}},\; E_{\text{vision}}\bigr] \in \mathbb{R}^{(L_{\text{text}} + L_{\text{audio}} + L_{\text{vision}}) \times d_{\text{model}}}
$$

---

## 3. Audio Transformer (AuT) Encoder

### 3.1 Architecture Overview

The Audio Transformer (AuT) is an **attention-encoder-decoder based auto-regressive model** designed to extract robust, general-purpose audio representations. Qwen3-Omni employs **only the encoder portion** as the audio encoder within the perceivation pipeline.

| Property | Specification |
|----------|--------------|
| Architecture Type | Attention-Encoder-Decoder (Autoregressive) |
| Component Used in Qwen3-Omni | Encoder only |
| Parameter Count | $\sim 0.6\text{B}$ (encoder) / $\sim 0.65\text{B}$ (full, per Table 1) |
| Pretraining Data | $20$ million hours of supervised audio |
| Input Features | 128-channel mel-spectrogram filter bank features |
| Output Token Rate | $12.5\text{ Hz}$ (one representation per $80\text{ ms}$) |
| Streaming Support | ✓ |

### 3.2 Internal Pipeline

```
Raw Audio Waveform
       │
       ▼
 Resample → 16 kHz
       │
       ▼
 128-Channel Mel-Spectrogram
 (25ms window, 10ms hop → 100 frames/s)
       │
       ▼
 ┌─────────────────────────┐
 │  Conv2D Downsampling     │  ← 8× temporal reduction
 │  (Sequential blocks)     │  → 12.5 frames/s
 └──────────┬──────────────┘
            │
            ▼
 ┌─────────────────────────┐
 │  Attention Layers        │  ← Flash Attention
 │  (Dynamic window 1–8s)   │     with dynamic window sizes
 └──────────┬──────────────┘
            │
            ▼
    E_audio ∈ ℝ^{L_audio × d_model}
```

### 3.3 Conv2D Downsampling Blocks

The downsampling precedes the attention modules:

- **Mechanism**: Sequential 2D convolutional blocks operating on the mel-spectrogram's time-frequency representation.
- **Reduction Factor**: $8\times$ temporal compression.
- **Purpose**: Reduces the computational burden on subsequent attention layers by compressing the $100\text{ Hz}$ mel-spectrogram frame rate to $12.5\text{ Hz}$.

$$
F_{\text{down}} = \text{Conv2D}_{\downarrow 8}(S_{\text{mel}}) \in \mathbb{R}^{L_{\text{audio}} \times d_{\text{conv}}}
$$

where $L_{\text{audio}} = \lceil T_{\text{frames}} / 8 \rceil$.

### 3.4 Attention Mechanism with Dynamic Windows

The AuT attention layers employ **flash attention** with **dynamic attention window sizes** spanning $1$ to $8$ seconds.

**Design Rationale**:

| Window Size | Strength | Use Case |
|-------------|----------|----------|
| Short ($1$–$2\text{s}$) | Efficient real-time prefill caching | Streaming / real-time interaction |
| Long ($4$–$8\text{s}$) | Deep contextual modeling | Offline audio understanding tasks |

The dynamic window balances:
- **Real-time prefill caching efficiency**: Short windows enable low-latency chunk-by-chunk processing.
- **Deep contextual offline modeling**: Long windows capture extended temporal dependencies for tasks such as audio event classification or long-form speech recognition.

### 3.5 Pretraining Data Composition

| Data Category | Proportion | Description |
|---------------|-----------|-------------|
| Chinese & English pseudo-labeled ASR | $80\%$ | Large-scale speech recognition |
| Other-language ASR | $10\%$ | Multilingual speech recognition |
| Audio understanding | $10\%$ | Non-speech audio events, environmental sounds, music |

**Total**: $20$ million hours of supervised audio data.

The mixture of ASR and audio understanding tasks ensures that AuT learns **general-purpose audio representations** rather than speech-only features.

---

## 4. Vision Encoder

### 4.1 Architecture

| Property | Specification |
|----------|--------------|
| Base Model | SigLIP2-So400m |
| Derived From | Qwen3-VL vision encoder |
| Parameter Count | $\sim 543\text{M}$ |
| Input Types | Image, Video (frames) |
| Training Data | Mixed image and video data |
| Streaming Support | — (chunked along temporal dimension for video) |

### 4.2 Image Processing

Static images are directly processed by the vision encoder:

$$
E_{\text{image}} = \text{VisionEncoder}(X_{\text{image}}) \in \mathbb{R}^{L_{\text{patches}} \times d_{\text{model}}}
$$

where $L_{\text{patches}}$ depends on the image resolution and patch size of the SigLIP2-So400m backbone.

### 4.3 Video Processing

For video inputs, the pipeline separates visual and auditory streams:

1. **Visual Track**: Frames are sampled at a **dynamic frame rate** designed to:
   - Maximize spatiotemporal variance (adaptive sampling, not fixed FPS).
   - Synchronize with the $12.5\text{ Hz}$ acoustic token rate from the AuT encoder.
   
2. **Audio Track**: Extracted and processed independently through the full AuT pipeline (§3).

$$
E_{\text{video\_visual}} = \text{VisionEncoder}\bigl(\text{DynamicSample}(X_{\text{video}})\bigr)
$$

$$
E_{\text{video\_audio}} = \text{AuT}_{\text{Encoder}}\bigl(\text{Conv2D}_{\downarrow 8}(\text{MelSpec}(\text{ExtractAudio}(X_{\text{video}})))\bigr)
$$

Both representations are temporally aligned through TM-RoPE (§5).

---

## 5. Time-aligned Multimodal Rotary Position Embedding (TM-RoPE)

### 5.1 Motivation

Standard Rotary Position Embedding (RoPE) operates in 1D, which is insufficient for multimodal sequences that span temporal, spatial (height, width), and cross-modal dimensions. Extended multimodal contexts suffer from **catastrophic sequence extrapolation degradation** if position encodings do not explicitly encode absolute temporal information across modalities.

TM-RoPE extends M-RoPE (Multimodal RoPE) by **incorporating absolute temporal information**, projecting temporal constraints onto 3D positional identifiers.

### 5.2 Dimension Factorization

The rotary embedding space is **orthogonally partitioned** into three components:

| Component | Symbol | Number of Rotary Angles | Function |
|-----------|--------|------------------------|----------|
| Temporal | $\theta_T$ | $24$ | Encodes absolute time position |
| Height | $\theta_H$ | $20$ | Encodes spatial row position |
| Width | $\theta_W$ | $20$ | Encodes spatial column position |
| **Total** | | **$64$** | Full rotary dimension |

The total rotary embedding dimensionality is $24 + 20 + 20 = 64$ angles.

**Key Design Decision**: The temporal component uses $24$ angles (allocated to higher-frequency rotary dimensions), providing **stronger oscillatory patterns** that capture fine-grained local temporal variation more effectively.

### 5.3 Modality-Specific Position ID Assignment

#### 5.3.1 Text Tokens

For text, height, width, and temporal components **share identical position identifiers**, mathematically collapsing TM-RoPE into standard 1D RoPE:

$$
\forall\; \text{token}_i \in E_{\text{text}}: \quad \text{ID}_T^{(i)} = \text{ID}_H^{(i)} = \text{ID}_W^{(i)} = \text{ID}_{\text{base}} + i
$$

This ensures backward compatibility: text-only operation is functionally identical to standard RoPE-equipped transformers.

#### 5.3.2 Audio Tokens

Audio tokens use shared position IDs across all three dimensions but are **strictly augmented with absolute temporal encodings**, advancing the temporal ID every $80\text{ ms}$ (matching the $12.5\text{ Hz}$ token rate):

$$
\forall\; \text{frame}_t \in E_{\text{audio}}: \quad \text{ID}_T^{(t)} = \text{ID}_H^{(t)} = \text{ID}_W^{(t)} = \text{ID}_{\text{base}} + \left\lfloor \frac{t_{\text{absolute}}}{80\text{ ms}} \right\rfloor
$$

Since audio has no spatial extent, $\text{ID}_H = \text{ID}_W = \text{ID}_T$, effectively reducing to 1D but with **absolute time anchoring** rather than relative sequential indexing.

#### 5.3.3 Vision Tokens — Images

For still images, the temporal ID is **constant** (single time point), while spatial tokens are distributed across row ($r$) and column ($c$) coordinates:

$$
\text{ID}_T = \text{const}, \quad \text{ID}_H = r, \quad \text{ID}_W = c
$$

#### 5.3.4 Vision Tokens — Video Frames

For video, temporal IDs **monotonically increase**, dynamically adjusted according to raw timestamps to maintain $80\text{ ms}$ resolution per ID. Within each frame, spatial tokens are distributed as for images:

$$
\text{ID}_T = f(t_{\text{absolute}}) = \text{ID}_{\text{base}} + \left\lfloor \frac{t_{\text{absolute}}}{80\text{ ms}} \right\rfloor, \quad \text{ID}_H = r, \quad \text{ID}_W = c
$$

This $80\text{ ms}$ temporal resolution is **identical** to the audio token rate, enabling direct temporal alignment between audio and video representations without fixed chunking.

### 5.4 Cross-Modal Contiguity Enforcement

To prevent positional conflicts when multiple modalities appear sequentially, each modality initializes its position indices at:

$$
\text{ID}_{\text{base}}^{(m)} = 1 + \max\bigl(\text{ID}_{\text{assigned}}^{(m-1)}\bigr)
$$

where $m$ indexes the sequential modality order within the input sequence.

**Consequence**: Audio and video representations are directly aligned via their temporal IDs, bypassing fixed chunking and facilitating **arbitrary streaming input durations**.

### 5.5 RoPE Application

Given a query/key vector $\mathbf{x} \in \mathbb{R}^{d}$ at position $(\text{ID}_T, \text{ID}_H, \text{ID}_W)$, the TM-RoPE rotation is applied as:

$$
\text{TM-RoPE}(\mathbf{x}) = \begin{bmatrix} R(\text{ID}_T, \theta_T) \cdot \mathbf{x}_{[1:48]} \\ R(\text{ID}_H, \theta_H) \cdot \mathbf{x}_{[49:88]} \\ R(\text{ID}_W, \theta_W) \cdot \mathbf{x}_{[89:128]} \end{bmatrix}
$$

where $R(\text{pos}, \theta)$ is the standard rotary matrix applied to the designated embedding dimensions, and the dimension indices correspond to the $24+20+20$ factorization (each angle occupying 2 real-valued dimensions, yielding $2 \times 64 = 128$ total rotary dimensions).

---

## 6. Thinker Module — Cognitive Reasoning Engine

### 6.1 Architecture

| Property | Specification |
|----------|--------------|
| Architecture | Mixture-of-Experts (MoE) Transformer |
| Total Parameters | $30\text{B}$ |
| Active Parameters per Token | $3\text{B}$ (A3B) |
| Input | TM-RoPE-encoded multimodal embeddings $E_{\text{multimodal}}$ |
| Output | (1) Text response tokens $Y_{\text{text}}$; (2) High-dimensional features $H_{\text{Thinker}}$ |
| Streaming Support | ✓ (chunked prefilling along temporal dimension) |

### 6.2 Functional Role

The Thinker serves as the **central cognitive reasoning engine** of Qwen3-Omni:

1. **Multimodal Comprehension**: Processes the unified multimodal embedding sequence to understand the user's intent across text, audio, image, and video.
2. **Text Generation**: Autoregressively generates text response tokens $Y_{\text{text}}$.
3. **Feature Provision**: Produces high-dimensional features $H_{\text{Thinker}}$ that encode multimodal context for the Talker.

$$
H_{\text{Thinker}}^{(N)},\; Y_{\text{text}}^{(N)} = \text{MoE}_{\text{Thinker}}\bigl(E_{\text{text}}^{(N)},\; E_{\text{audio}}^{(N)},\; E_{\text{vision}}^{(N)}\bigr)
$$

### 6.3 MoE Advantages for the Thinker

- **Throughput**: Activating only $3\text{B}$ of $30\text{B}$ parameters per token drastically reduces FLOPs compared to a dense $30\text{B}$ model.
- **KV-Cache Efficiency**: MoE reduces I/O consumption from KV cache during long-sequence processing, increasing tokens per second (TPS).
- **Concurrency**: Under high-concurrency scenarios, prefill latency and TTFT remain largely unaffected (see §12).

### 6.4 Independent System Prompt

Because textual representations are **decoupled** from the Talker, the Thinker operates under its **own system prompt**, independently controlling:
- Response style (formal, casual, concise, verbose, etc.)
- Reasoning depth
- Language selection

---

## 7. Talker Module — Streaming Acoustic Generator

### 7.1 Architecture

| Property | Specification |
|----------|--------------|
| Architecture | Mixture-of-Experts (MoE) Transformer |
| Total Parameters | $3\text{B}$ |
| Active Parameters per Token | $0.3\text{B}$ (A0.3B) |
| Input | Audio + Vision multimodal features from Thinker; conversational history; text tokens (via preprocessing) |
| Output | Primary codebook token $C_0^{(t)}$ per frame |
| Streaming Support | ✓ |
| Generation Rate | $12.5\text{ Hz}$ (one codec frame per $80\text{ ms}$ of synthesized audio) |

### 7.2 Critical Design: Decoupled Conditioning

The Talker **does not consume** the Thinker's high-level text representations. Instead, it conditions on:

1. **Audio multimodal features** ($E_{\text{audio\_ctx}}$) from the Thinker's hidden states.
2. **Vision multimodal features** ($E_{\text{vision\_ctx}}$) from the Thinker's hidden states.
3. **Text tokens** supplied via controlled preprocessing (not internal text embeddings).

$$
H_{\text{Talker}}^{(N-1)} = \text{MoE}_{\text{Talker}}\bigl(H_{\text{Thinker, audio+vision}}^{(N-1)},\; E_{\text{audio\_ctx}},\; E_{\text{vision\_ctx}}\bigr)
$$

**Rationale**:
- Discrete text tokens and embeddings are **information-equivalent** for textual content — no information is lost by decoupling.
- Multimodal conditioning is **necessary** for preserving prosody, timbre, and acoustic dynamics in tasks like speech translation.
- Decoupling permits **external intervention**: RAG systems, function-calling modules, and safety filters can modify the Thinker's text output before optionally feeding it to the Talker.

### 7.3 Primary Codebook Generation

At each temporal step $t$ (at $12.5\text{ Hz}$), the Talker produces the **zeroth codebook token**:

$$
C_0^{(t)} = \text{LinearHead}\bigl(H_{\text{Talker}}^{(t)}\bigr)
$$

This token represents the coarsest level of the Residual Vector Quantization (RVQ) codec representation.

### 7.4 Independent System Prompt

The Talker has its **own system prompt**, independently controlling:
- Voice style (pitch, speed, expressiveness)
- Audio characteristics (warmth, clarity)
- Language-specific pronunciation rules

---

## 8. Multi-Token Prediction (MTP) Module

### 8.1 Architecture

| Property | Specification |
|----------|--------------|
| Architecture | Dense Transformer (NOT MoE) |
| Parameter Count | $80\text{M}$ |
| Function | Predict residual codebook tokens $C_{1 \dots K}^{(t)}$ given $C_0^{(t)}$ |
| Autoregressive Mode | Fixed-step (deterministic number of steps per frame) |
| Context | Left-context only (strictly causal) |
| Streaming Support | ✓ |

### 8.2 Operational Details

Once the Talker generates $C_0^{(t)}$ for frame $t$, the MTP module autoregressively generates the remaining $K$ residual codebook tokens for that frame:

$$
C_{1 \dots K}^{(t)} = \text{MTP}\bigl(C_0^{(t)},\; \text{Context}_{<t}\bigr)
$$

**Key Properties**:

- **Left-context-only attention**: The MTP module attends exclusively to past context, enforcing strict causal streaming constraints. No future information is required.
- **Fixed-step autoregression**: The number of residual codebooks $K$ is fixed, enabling a **fixed KV cache memory space** for acceleration.
- **Ultra-lightweight**: At $80\text{M}$ parameters, the MTP module has minimal computational FLOPs and low memory bandwidth requirements, enabling efficient batch processing.

### 8.3 Advantage over Block-Context DiT

In the predecessor (Qwen2.5-Omni), the system required waiting for **sufficient block-context** from the Talker before synthesis could begin. With the per-frame MTP approach in Qwen3-Omni:

- Waveform can be output **immediately** after the Talker generates each token.
- First-packet latency is **significantly reduced**.
- No block-size dependency exists in the generation pipeline.

---

## 9. Code2Wav ConvNet Decoder

### 9.1 Architecture

| Property | Specification |
|----------|--------------|
| Architecture | Causal Convolutional Neural Network (ConvNet) |
| Parameter Count | $200\text{M}$ |
| Input | Multi-codebook sequence $C_{0 \dots K}^{(t)}$ for frame $t$ |
| Output | Continuous waveform segment $\hat{W}^{(t)}$ ($80\text{ ms}$ of audio) |
| Causality | Strictly causal (left-context only) |
| Streaming Support | ✓ |

### 9.2 Waveform Rendering

The ConvNet acts as a **neural vocoder**, directly synthesizing the continuous waveform from the RVQ codebook tokens:

$$
\hat{W}^{(t)} = \text{Code2Wav}_{\text{ConvNet}}\bigl(C_{0 \dots K}^{(t)}\bigr)
$$

### 9.3 Design Advantages

| Property | Benefit |
|----------|---------|
| **Causal convolution** | No future context needed → immediate per-frame rendering → streaming compatible |
| **Lightweight** ($200\text{M}$) | Low inference latency and computational cost (FLOPs) |
| **ConvNet architecture** | Extensive hardware acceleration support (CUDA, TensorRT, etc.) across diverse platforms |
| **Batched inference** | Efficient batch processing for high-concurrency scenarios |
| **Superior fidelity** | Achieves better audio fidelity compared to more complex DiT-based vocoders at lower cost |

---

## 10. Asynchronous Chunked Prefilling and Streaming Pipeline

### 10.1 Chunked Prefilling Mechanism

Both the audio and vision encoders output representations **chunked along the temporal dimension**. During real-time interaction, the Thinker and Talker perform **asynchronous prefilling**:

```
Time →
┌─────────────────────────────────────────────────────────────┐
│  Thinker Prefill Chunk N    │  Thinker Prefill Chunk N+1   │
│  (processes multimodal      │  (processes next chunk)       │
│   input chunk N)            │                               │
└──────────┬──────────────────┴───────────────────────────────┘
           │ H_Thinker^(N) ready
           ▼
┌──────────────────────────────┐
│  Talker Prefill Chunk N      │  (runs ASYNCHRONOUSLY while
│  (conditions on H_Thinker^N) │   Thinker processes chunk N+1)
└──────────┬───────────────────┘
           │ C_0^(t) generated
           ▼
┌──────────────────────────────┐
│  MTP: Generate C_{1..K}^(t)  │  (immediate, per-frame)
└──────────┬───────────────────┘
           │
           ▼
┌──────────────────────────────┐
│  Code2Wav: Render Ŵ^(t)     │  (immediate, per-frame)
└──────────┬───────────────────┘
           │
           ▼
   Streaming Audio Buffer → User
```

**Concurrency Benefit**: When Thinker completes prefilling chunk $N$, its output is immediately used to prefill the Talker for chunk $N$, while Thinker simultaneously begins prefilling chunk $N+1$. This overlap **significantly reduces TTFT** for both modules.

### 10.2 Sequential Dependency Chain

The total first-packet latency is the **sum** of sequential component latencies:

$$
\text{Latency}_{\text{total}} = \text{Latency}_{\text{preprocess}} + \text{TTFT}_{\text{Thinker}} + \text{TTFT}_{\text{Talker}} + \text{Time}_{\text{MTP}} + \text{Time}_{\text{Code2Wav}}
$$

Each component in the chain depends on the output of the preceding component.

---

## 11. End-to-End Algorithmic Pipeline (Unified Formal Description)

The following presents the complete, unified algorithmic description of Qwen3-Omni's end-to-end execution path.

---

### Algorithm: Qwen3-Omni End-to-End Inference

**Input**: Raw multimodal data: $X_{\text{text}}$, $X_{\text{audio}}$, $X_{\text{image}}$, $X_{\text{video}}$

**Output**: Text response $Y_{\text{text}}$ and streaming waveform $\hat{W}$

---

**Phase 1: Perceivation — Multimodal Feature Extraction**

**Step 1.1 — Text Encoding**:

$$
E_{\text{text}} = \text{Embedding}\bigl(\text{BPE}_{V=151643}(X_{\text{text}})\bigr) \in \mathbb{R}^{L_{\text{text}} \times d}
$$

**Step 1.2 — Audio Encoding**:

$$
S_{\text{mel}} = \text{MelSpectrogram}_{128\text{ch},\,25\text{ms},\,10\text{ms}}(\text{Resample}_{16\text{kHz}}(X_{\text{audio}}))
$$

$$
E_{\text{audio}} = \text{AuT}_{\text{Encoder}}^{0.6\text{B}}\bigl(\text{Conv2D}_{\downarrow 8}(S_{\text{mel}})\bigr) \in \mathbb{R}^{L_{\text{audio}} \times d}
$$

**Step 1.3 — Vision Encoding**:

$$
E_{\text{vision}} = \text{VisionEncoder}_{543\text{M}}^{\text{SigLIP2}}\bigl(\text{DynamicSample}(X_{\text{vision}})\bigr) \in \mathbb{R}^{L_{\text{vision}} \times d}
$$

For video: additionally extract audio track and process via Step 1.2.

---

**Phase 2: Positional Encoding — TM-RoPE Assignment**

For each token in $E_{\text{multimodal}} = [E_{\text{text}}; E_{\text{audio}}; E_{\text{vision}}]$:

**Step 2.1 — Cross-Modal Base ID**:

$$
\text{ID}_{\text{base}}^{(m)} = 1 + \max\bigl(\text{ID}_{\text{assigned}}^{(m-1)}\bigr)
$$

**Step 2.2 — Text IDs** (collapse to 1D RoPE):

$$
\text{ID}_T^{(i)} = \text{ID}_H^{(i)} = \text{ID}_W^{(i)} = \text{ID}_{\text{base}} + i
$$

**Step 2.3 — Audio IDs** (absolute $80\text{ ms}$ temporal anchoring):

$$
\text{ID}_T^{(t)} = \text{ID}_H^{(t)} = \text{ID}_W^{(t)} = \text{ID}_{\text{base}} + \left\lfloor \frac{t_{\text{abs}}}{80\text{ ms}} \right\rfloor
$$

**Step 2.4 — Vision IDs**:

- Image: $\text{ID}_T = \text{const},\; \text{ID}_H = r,\; \text{ID}_W = c$
- Video: $\text{ID}_T = \text{ID}_{\text{base}} + \lfloor t_{\text{abs}} / 80\text{ ms} \rfloor,\; \text{ID}_H = r,\; \text{ID}_W = c$

**Step 2.5 — Apply Rotary Matrices** ($24 + 20 + 20$ angles):

$$
\tilde{E}_{\text{multimodal}} = \text{TM-RoPE}\bigl(E_{\text{multimodal}},\; \{\text{ID}_T, \text{ID}_H, \text{ID}_W\}\bigr)
$$

---

**Phase 3: Thinker — Cognitive Reasoning (Asynchronous Chunk $N$)**

$$
H_{\text{Thinker}}^{(N)},\; Y_{\text{text}}^{(N)} = \text{MoE}_{\text{Thinker}}^{30\text{B-A3B}}\bigl(\tilde{E}_{\text{multimodal}}^{(N)}\bigr)
$$

Output: Text tokens $Y_{\text{text}}^{(N)}$ and high-dimensional features $H_{\text{Thinker}}^{(N)}$.

---

**Phase 4: Talker — Acoustic Generation (Asynchronous Chunk $N-1$)**

Conditions on **audio and vision features only** (not text representations):

$$
H_{\text{Talker}}^{(N-1)} = \text{MoE}_{\text{Talker}}^{3\text{B-A0.3B}}\bigl(H_{\text{Thinker, audio+vision}}^{(N-1)},\; E_{\text{audio\_ctx}},\; E_{\text{vision\_ctx}}\bigr)
$$

Generate primary codebook token for frame $t$:

$$
C_0^{(t)} = \text{LinearHead}\bigl(H_{\text{Talker}}^{(t)}\bigr)
$$

---

**Phase 5: MTP — Residual Codebook Prediction (Per-Frame)**

$$
C_{1 \dots K}^{(t)} = \text{MTP}^{80\text{M}}_{\text{DenseTransformer}}\bigl(C_0^{(t)},\; \text{Context}_{<t}\bigr)
$$

Left-context-only; fixed-step autoregression; fixed KV cache.

---

**Phase 6: Code2Wav — Waveform Synthesis (Per-Frame)**

$$
\hat{W}^{(t)} = \text{Code2Wav}^{200\text{M}}_{\text{CausalConvNet}}\bigl(C_{0 \dots K}^{(t)}\bigr)
$$

Yield $\hat{W}^{(t)}$ to streaming buffer immediately.

---

**Phase 7: Streaming Output**

Concatenate $\hat{W}^{(t)}$ for $t = 1, 2, \dots$ into continuous audio stream. Each frame represents $80\text{ ms}$ of synthesized audio. First-packet latency: $234\text{ ms}$ (audio) / $547\text{ ms}$ (video) from cold start at $1\times$ concurrency.

---

## 12. Latency Analysis and Concurrency Characteristics

### 12.1 First-Packet Latency Decomposition

| Component | 1 Concurrency (Audio/Video) | 4 Concurrency (Audio/Video) | 6 Concurrency (Audio/Video) |
|-----------|----------------------------|----------------------------|----------------------------|
| Thinker-Talker Tail Packet Preprocessing | $72 / 160\text{ ms}$ | $94 / 180\text{ ms}$ | $100 / 200\text{ ms}$ |
| Thinker TTFT | $88 / 160\text{ ms}$ | $468 / 866\text{ ms}$ | $673 / 1330\text{ ms}$ |
| Talker TTFT | $57 / 210\text{ ms}$ | $145 / 450\text{ ms}$ | $376 / 734\text{ ms}$ |
| MTP Module (per token) | $14\text{ ms}$ | $16\text{ ms}$ | $18\text{ ms}$ |
| Code2Wav (per code) | $3\text{ ms}$ | $5\text{ ms}$ | $5\text{ ms}$ |
| **Overall Latency** | **$234 / 547\text{ ms}$** | **$728 / 1517\text{ ms}$** | **$1172 / 2284\text{ ms}$** |

### 12.2 Generation Throughput

| Metric | 1 Concurrency | 4 Concurrency | 6 Concurrency |
|--------|--------------|--------------|--------------|
| Thinker TPS | $75\text{ tokens/s}$ | $63\text{ tokens/s}$ | $53\text{ tokens/s}$ |
| Talker TPS | $140\text{ tokens/s}$ | $125\text{ tokens/s}$ | $110\text{ tokens/s}$ |
| Generation RTF | $0.47$ | $0.56$ | $0.66$ |

### 12.3 Real-Time Factor (RTF) Computation

After the initial packet is output and streaming synthesis begins, the Talker at $12.5\text{ Hz}$ requires only one token to synthesize $80\text{ ms}$ of audio. The RTF is:

$$
\text{RTF} = \frac{T_{\text{Thinker}}^{\text{1 token}} + T_{\text{Talker}}^{\text{1 token}} + T_{\text{MTP}}^{\text{1 token}} + T_{\text{Code2Wav}}^{\text{1 code}}}{80\text{ ms}}
$$

**Critical Observation**: RTF remains **below $1.0$** across all concurrency levels ($0.47$, $0.56$, $0.66$), guaranteeing that users receive continuously streaming audio responses without buffering gaps.

### 12.4 Concurrency Resilience of MoE

The MoE architecture ensures that:
- **Prefill latency** scales sub-linearly with concurrency (only active parameters are computed per token).
- **KV cache I/O** is reduced compared to dense models, as the effective model width per token is smaller.
- **MTP and Code2Wav** latencies remain nearly constant across concurrency levels due to their lightweight, batch-friendly architectures.

---

## 13. Complete Module Summary Table

| Module | Architecture | Parameters | Input | Output | Token Rate | Streaming | Key Innovation |
|--------|-------------|------------|-------|--------|-----------|-----------|----------------|
| **BPE Tokenizer** | Byte-level BPE | — | Raw text | Token IDs ($V=151{,}643$) | — | — | Lossless Unicode coverage |
| **Text Embedding** | Lookup table | Part of Thinker | Token IDs | $E_{\text{text}} \in \mathbb{R}^{L \times d}$ | — | — | — |
| **Audio Preprocessing** | Signal processing | — | Raw waveform | 128-ch mel-spectrogram | $100\text{ Hz}$ | ✓ | $16\text{ kHz}$, $25\text{ ms}$ window, $10\text{ ms}$ hop |
| **Conv2D Downsampler** | Sequential Conv2D | Part of AuT | Mel-spectrogram | Compressed features | $12.5\text{ Hz}$ | ✓ | $8\times$ temporal reduction |
| **AuT Encoder** | Attention Transformer | $\sim 0.6\text{B}$ ($650\text{M}$) | Compressed features | $E_{\text{audio}} \in \mathbb{R}^{L \times d}$ | $12.5\text{ Hz}$ | ✓ | Dynamic attention window $1$–$8\text{s}$; $20\text{M}$ hrs pretraining |
| **Vision Encoder** | SigLIP2-So400m | $\sim 543\text{M}$ ($540\text{M}$) | Image/Video frames | $E_{\text{vision}} \in \mathbb{R}^{L \times d}$ | Dynamic FPS | — | Mixed image+video training; from Qwen3-VL |
| **TM-RoPE** | Positional encoding | — | Position indices | Rotary matrices ($24{+}20{+}20$) | — | — | Absolute temporal alignment; $80\text{ ms}$ resolution |
| **Thinker** | MoE Transformer | $30\text{B}$-A$3\text{B}$ | $\tilde{E}_{\text{multimodal}}$ | $Y_{\text{text}}$, $H_{\text{Thinker}}$ | — | ✓ | Cognitive reasoning; independent system prompt |
| **Talker** | MoE Transformer | $3\text{B}$-A$0.3\text{B}$ | Audio+Vision features | $C_0^{(t)}$ | $12.5\text{ Hz}$ | ✓ | Decoupled from text repr.; independent system prompt |
| **MTP** | Dense Transformer | $80\text{M}$ | $C_0^{(t)}$, left context | $C_{1 \dots K}^{(t)}$ | $12.5\text{ Hz}$ | ✓ | Fixed-step AR; fixed KV cache; ultra-lightweight |
| **Code2Wav** | Causal ConvNet | $200\text{M}$ | $C_{0 \dots K}^{(t)}$ | $\hat{W}^{(t)}$ (waveform) | $12.5\text{ Hz}$ | ✓ | Immediate per-frame rendering; no block context needed |

**Total System First-Packet Latency**: $234\text{ ms}$ (audio) / $547\text{ ms}$ (video) at $1\times$ concurrency.

**Generation RTF**: $0.47$ at $1\times$ concurrency (real-time streaming guaranteed with $\text{RTF} < 1$).

---

*This document constitutes the complete end-to-end architectural specification of Qwen3-Omni, covering every module from raw input perceivation through multimodal encoding, positional embedding, cognitive reasoning, acoustic generation, codec prediction, waveform synthesis, and streaming output — with all mathematical formulations, parameter counts, design rationale, and latency characteristics consolidated in a single reference.*